[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q2/03_sanity_checks_and_gradcam.ipynb)

# R2-Q2 Week 3 — Sanity-check Grad-CAM, then run it on the misclassification set

This notebook does two things in order. First, it runs Adebayo et al. (2018)'s sanity checks on vanilla Grad-CAM applied to the R2-Q1 classifier — the model-parameter randomization check and the data-randomization check — and reports the Spearman rank correlation between trained-model and randomized-model heatmaps at each randomization stage, with ρ < 0.3 at full randomization committed as the passing criterion. Second, once vanilla Grad-CAM has passed the sanity checks, it generates heatmaps for every image in the working set produced by Week 2 and saves them for Week 4's categorization work.

By the end of this notebook you will have:

- A `sanity_check_results.json` recording the Spearman ρ trajectory across randomization stages for both the model-parameter and data-randomization checks, the verdict against the ρ < 0.3 passing criterion, and the methods-section text justifying that the trained model's downstream interpretation rests on a sanity-checked saliency method.
- A `heatmaps/` directory containing one `.npy` file per misclassification — 81 files, each a 7×7 numpy array (pre-upsampling) — plus a `gradcam_metadata.parquet` index that joins the heatmap files back to the working set on `filename`. Week 4 upsamples each heatmap at use time.
- A `shuffled_resnet18.pt` checkpoint — the label-shuffled-trained classifier used for the data-randomization sanity check, saved so the check is reproducible without retraining.
- A diagnostic figure (`sanity_check_trajectory.png`) showing the Spearman ρ curve across randomization stages, so a reader can see the trajectory the verdict was made from, not just the verdict itself.

## Before you run anything: switch to a GPU runtime

This notebook trains a neural network, which needs a more powerful processing 
unit (GPU) to work. By default, Colab gives you a CPU-only runtime — fine for 
last week's notebooks, but it won't work for this one.

To switch:

1. In Colab's menu bar: **Runtime → Change runtime type**.
2. Under "Hardware accelerator," choose **T4 GPU**.
3. Click **Save**.

Wait for the session to come up on the new runtime (a few seconds).
Then run the cells below in order.

**If Colab tells you "no GPUs available right now":** free-tier GPU
access is shared and occasionally unavailable. Wait a few minutes
and try again. If it keeps refusing, tell your mentor — the
upgraded Colab tiers have more reliable GPU access.

In [ ]:
try:
    import irilab2026 as iri
    print(f"OK — irilab2026 {iri.__version__} is installed and all deps importable.")
    print("    → use Pattern 2 in the following Cell:")
    print("      !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main")
except ModuleNotFoundError as e:
    print(f"MISSING — {e.name} not importable.")
    print("    → use Pattern 1 in the following Cell:")
    print("      !pip install git+https://github.com/geraldmc/irilab2026.git@main")

In [ ]:
# Pattern 1 — fresh or partial runtime (installs deps that aren't present yet)
# This skips anything pip already sees as installed. This is ideal for a new environment or when you want
# to be sure everything is up to date.

# !pip install git+https://github.com/geraldmc/irilab2026.git@main

# Pattern 2 — populated runtime (refreshes irilab2026 only, leaves deps alone)
# This is ideal for when you already have the dependencies installed and just want to update irilab2026.

# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

### Confirm the GPU is in place

Before going further, confirm Colab attached a GPU. The cell below
prints `GPU detected:` followed by `True` or `False`. If it's `True`,
you're good. If it's `False`, go back to the top of the notebook and
re-do the runtime switch — the next cells will not work without a GPU.

The check is a separate cell from the main setup (below) because it's
fast and recoverable: if the GPU isn't there, you see the problem
right away with no Drive authorization prompt to dismiss first.

In [ ]:
import irilab2026 as iri

gpu_ok = iri.has_gpu()
print(f"GPU detected: {gpu_ok}")

assert gpu_ok, (
    "No GPU detected. Go to the top of the notebook and follow the "
    "runtime-switch instructions, then re-run from Setup Step 1."
)

In [ ]:
# Mount Drive, set up irilab2026 with a GPU requirement, seed everything,
# and point OUTPUT_DIR at the R2-Q1 output directory.
iri.setup(gpu_required=True)
iri.seed_all(42)

OUTPUT_DIR = iri.output_dir("r2_q2")
print(f"Output directory: {OUTPUT_DIR}")

## 2) Load the Week 1 pre-commit

This notebook does not read any specific field from `precommit.json`,
but the file is N01's signed deliverable, and downstream sanity-check
interpretation and Week-4 categorization both depend on it conceptually.
Loading it here is a guardrail: if the file is missing or malformed,
the cell below fails with a clear pointer back to N01 rather than
letting the notebook proceed into Grad-CAM work it can't actually
finish.

The load also light-validates the precommit's top-level structure —
the four keys `taxonomy`, `categorization_procedure`, `sanity_checks`,
and `aggregation_level` must all be present. If any are missing, the
precommit was probably written by an incomplete N01 run, and re-running
N01 to completion is the remedy.

In [ ]:
### 2.1) Load and validate the pre-commit

import json

precommit_path = OUTPUT_DIR / "precommit.json"

try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {precommit_path}. "
        "This file is produced by 01_pre_commits.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light validation: the four locked top-level keys must all be present.
expected_keys = {
    "taxonomy",
    "categorization_procedure",
    "sanity_checks",
    "aggregation_level",
}
missing = expected_keys - set(precommit.keys())
if missing:
    raise KeyError(
        f"precommit.json is missing top-level keys: {sorted(missing)}. "
        "This usually means N01 didn't complete — "
        "re-run 01_pre_commits.ipynb to regenerate the file."
    )

print(f"Loaded: {precommit_path}")
print(f"  top-level keys: {sorted(precommit.keys())}")

### 2.2) Load the working set

N02 wrote `misclassifications.parquet` to this question's output directory:
the subset of the PlantDoc prediction table where the model misclassified
at the category level. This is the working set every Grad-CAM pass in
Sections 3, 4, and 5 runs over.

The schema validation below is minimal — it confirms that the two columns
this notebook actually uses are present: `filename` (the per-image
identifier for the heatmap files Section 5 writes) and `pred_idx` (the
Grad-CAM target class for every pass). A missing column means N02 wrote
an older schema; re-running N02 is the remedy.

In [ ]:
### 2.2) Load the working set

import pandas as pd

working_set_path = OUTPUT_DIR / "misclassifications.parquet"

try:
    working_set = pd.read_parquet(working_set_path)
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {working_set_path}. "
        "This file is produced by 02_load_and_filter.ipynb — "
        "run that notebook to completion before running this one."
    ) from None

# Light schema validation: the columns this notebook actually uses must
# be present. `filename` is the per-image identifier that joins heatmap
# files back to working-set rows in Section 5; `pred_idx` is the Grad-CAM
# target class throughout Sections 3–5.
required_columns = {"filename", "pred_idx"}
missing = required_columns - set(working_set.columns)
if missing:
    raise KeyError(
        f"misclassifications.parquet is missing required columns: {sorted(missing)}. "
        "This usually means N02 wrote an older or partial schema — "
        "re-run 02_load_and_filter.ipynb to regenerate the file."
    )

# An empty working set means there are no misclassifications to inspect —
# methodologically interesting but operationally a stop condition for N03.
if len(working_set) == 0:
    raise ValueError(
        "misclassifications.parquet has zero rows. "
        "There are no misclassifications to inspect — "
        "if this is unexpected, check N02's filter logic."
    )

print(f"Loaded: {working_set_path}")
print(f"  rows: {len(working_set)}")
print(f"  columns: {sorted(working_set.columns)}")

### 2.3) Load the classifier and check it matches the working set

R2-Q1 produced the trained classifier this question runs on top of. It
lives in R2-Q1's output directory, not R2-Q2's — the cell below points
at it explicitly and loads it.

Two non-obvious choices in the code below:

First, `num_classes` is derived from the saved state_dict's `fc.weight`
shape, not hardcoded as 38 or read from a precommit. This keeps N03
decoupled from how R2-Q1 happened to number its classes — if a future
R2-Q1 run trains against a different class count, this notebook still
loads the model correctly.

Second, after loading, the cell cross-checks that the working set's max
`pred_idx` is less than the model's `num_classes`. If it isn't, the
working set was built against a different model than the one just loaded,
and any Grad-CAM target class index downstream would be meaningless.
Catching this here is far less painful than chasing a silent
miscategorization in Week 4.

The cell finishes with a forward pass on a dummy input — a small
assurance that the model loaded structurally cleanly and produces logits
of the expected shape.

In [ ]:
### 2.3) Load the classifier and check it matches the working set

import torch

# R2-Q1's output directory holds the trained classifier this notebook inherits.
R2Q1_DIR = iri.output_dir("r2_q1")
classifier_path = R2Q1_DIR / "baseline_resnet18.pt"

try:
    state_dict = torch.load(classifier_path, weights_only=True, map_location="cpu")
except FileNotFoundError:
    raise FileNotFoundError(
        f"Could not find {classifier_path}. "
        "This file is produced by R2-Q1's 03_baseline_classifier.ipynb — "
        "run that notebook in your R2-Q1 series first, "
        "or ensure your R2-Q1 output directory is mounted."
    ) from None

# Derive num_classes from the saved head's output shape. ResNet-18's
# fc.weight is (num_classes, 512), so its first dimension is the class
# count the saved model was trained against.
num_classes = state_dict["fc.weight"].shape[0]
print(f"num_classes (from state_dict fc.weight): {num_classes}")

# Build the matching architecture and load the trained weights.
# pretrained=False because we're loading trained weights anyway — no
# point downloading the ImageNet head we're about to overwrite.
model = iri.build_baseline_model(num_classes, pretrained=False)
model.load_state_dict(state_dict)
model = model.cuda().eval()

# Cross-check: every working-set row has a pred_idx that must be a valid
# class index for this model. If the max pred_idx is not less than
# num_classes, the working set and classifier were produced from
# different configurations and one of them needs to be regenerated.
max_pred_idx = int(working_set["pred_idx"].max())
assert max_pred_idx < num_classes, (
    f"working set pred_idx max ({max_pred_idx}) is not less than "
    f"the loaded model's num_classes ({num_classes}). "
    "The working set and classifier were produced from different "
    "configurations; one of them needs to be regenerated."
)
print(f"pred_idx max ({max_pred_idx}) < num_classes ({num_classes}) — OK")

# Smoke check: a forward pass on a dummy input confirms the loaded model
# is structurally sound and produces logits of the expected shape.
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device="cuda")
    out = model(dummy)
assert out.shape == (1, num_classes), (
    f"forward pass produced output shape {tuple(out.shape)}, "
    f"expected (1, {num_classes})."
)
print(f"smoke-check forward pass output shape: {tuple(out.shape)} — OK")